# PALLAS — Campaign B Trajectory Prediction

## Objective

Develop a machine-learning surrogate model to predict the
evolution of electron-beam properties along the accelerator.

### Inputs
- P_max
- cN2_max
- L_inj
- dip_frac
- x_of
- z_mm

### Outputs
- E_MeV
- q_pC
- dE_pct
- emit_um
- div_mrad
- sigz_um

We first establish a Linear Regression baseline and then
investigate nonlinear machine-learning models.

The primary evaluation concerns the in-plasma region.

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

## 1. Load Campaign B Trajectory Data

In [14]:
# Dataset location
DATA_PATH = Path("data/pallas_hackathon_data")

# Load Campaign B trajectories
T = pd.read_parquet(
    DATA_PATH / "campaign_B_trajectories.parquet"
)

# Inspect the dataset
print("Dataset shape:", T.shape)

print("\nAvailable columns:")
print(T.columns.tolist())

print("\nFirst five rows:")
display(T.head())

Dataset shape: (322615, 13)

Available columns:
['config', 'z_mm', 'x_of', 'P_max', 'cN2_max', 'L_inj', 'dip_frac', 'E_MeV', 'q_pC', 'dE_pct', 'emit_um', 'div_mrad', 'sigz_um']

First five rows:


,config,z_mm,x_of,P_max,cN2_max,L_inj,dip_frac,E_MeV,q_pC,dE_pct,emit_um,div_mrad,sigz_um
0,1000,2.015055,1277.489014,6886.891113,0.247778,0.601816,0.430989,32.233013,124.356285,8.757664,5.606892,34.826664,3.354679
1,1000,2.060053,1277.489014,6886.891113,0.247778,0.601816,0.430989,35.585037,125.272934,7.883402,7.428158,31.144773,3.428018
2,1000,2.105055,1277.489014,6886.891113,0.247778,0.601816,0.430989,39.021626,125.180725,7.409557,9.591781,29.758060,3.508575
3,1000,2.150053,1277.489014,6886.891113,0.247778,0.601816,0.430989,42.519985,125.177795,7.339853,12.299101,28.584585,3.594396
4,1000,2.195155,1277.489014,6886.891113,0.247778,0.601816,0.430989,46.158367,125.177788,7.402719,15.091762,26.663004,3.682620


## 2. Data Preparation

In [15]:
# Define input and output variables
FEATURES = [
    "P_max", "cN2_max", "L_inj",
    "dip_frac", "x_of", "z_mm"
]

TARGETS = [
    "E_MeV", "q_pC", "dE_pct",
    "emit_um", "div_mrad", "sigz_um"
]

# Count simulations
print("Number of simulations:", T["config"].nunique())

# Number of observation points per simulation
print("\nPoints per simulation:")
print(T.groupby("config").size().describe())

# Check missing values
print("\nMissing values:")
print(T[FEATURES + TARGETS].isna().sum())

# Check longitudinal range
print("\nLongitudinal range:")
print("Minimum z (mm):", T["z_mm"].min())
print("Maximum z (mm):", T["z_mm"].max())

Number of simulations: 2855

Points per simulation:
count    2855.0
mean      113.0
std         0.0
min       113.0
25%       113.0
50%       113.0
75%       113.0
max       113.0
dtype: float64

Missing values:
P_max       0
cN2_max     0
L_inj       0
dip_frac    0
x_of        0
z_mm        0
E_MeV       0
q_pC        0
dE_pct      0
emit_um     0
div_mrad    0
sigz_um     0
dtype: int64

Longitudinal range:
Minimum z (mm): 2.015055
Maximum z (mm): 7.053853


## 3. Training and Validation Split

In [16]:
from sklearn.model_selection import GroupShuffleSplit

# Remove rows with missing input/output values
T_clean = T.dropna(subset=FEATURES + TARGETS).copy()

# Define inputs, outputs, and simulation groups
X = T_clean[FEATURES]
y = T_clean[TARGETS]
groups = T_clean["config"]

# Split by simulation configuration
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

# Check the split
train_configs = set(groups.iloc[train_idx])
val_configs = set(groups.iloc[val_idx])

print("Training simulations:", len(train_configs))
print("Validation simulations:", len(val_configs))

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))

print("Overlapping simulations:", len(train_configs & val_configs))

Training simulations: 2284
Validation simulations: 571
Training rows: 258092
Validation rows: 64523
Overlapping simulations: 0


## 4. Linear Regression Baseline

In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

# Define the linear regression model
linear_model = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

# Train the model
linear_model.fit(X_train, y_train)

# Predict beam properties
y_pred = linear_model.predict(X_val)

# Calculate R² for each output
linear_scores = {}

for i, target in enumerate(TARGETS):
    linear_scores[target] = r2_score(
        y_val[target],
        y_pred[:, i]
    )

# Calculate mean R²
linear_scores["mean"] = np.mean(
    [linear_scores[target] for target in TARGETS]
)

# Display results
display(
    pd.DataFrame.from_dict(
        linear_scores,
        orient="index",
        columns=["R²"]
    ).round(4)
)

,R²
E_MeV,0.6974
q_pC,0.9269
dE_pct,0.3066
emit_um,0.3575
div_mrad,0.6679
sigz_um,0.5948
mean,0.5919


## 5. In-Plasma Evaluation

In [18]:
# Create a DataFrame with validation inputs and true outputs
eval_df = X_val.copy()

for target in TARGETS:
    eval_df[target] = y_val[target]

# Add predicted values
for i, target in enumerate(TARGETS):
    eval_df[f"pred_{target}"] = y_pred[:, i]

# Identify in-plasma rows
eval_df["in_plasma"] = (
    eval_df["z_mm"] <= eval_df["L_inj"] + 3.2
)

# Keep only in-plasma data
plasma_df = eval_df[eval_df["in_plasma"]].copy()

print("Validation rows:", len(eval_df))
print("In-plasma rows:", len(plasma_df))

# Calculate R² at each longitudinal position
# Positions are grouped by their index within each simulation
plasma_df["position_index"] = (
    plasma_df.groupby(
        [plasma_df.index.map(lambda idx: T_clean.loc[idx, "config"])]
    ).cumcount()
)

position_scores = []

for position, data in plasma_df.groupby("position_index"):

    if len(data) < 2:
        continue

    scores = {
        "position": position
    }

    for target in TARGETS:
        scores[target] = r2_score(
            data[target],
            data[f"pred_{target}"]
        )

    position_scores.append(scores)

position_df = pd.DataFrame(position_scores)

# Mean R² across positions
plasma_scores = position_df[TARGETS].mean()

print("\nIn-plasma R² per beam property:")
display(plasma_scores.round(4))

print("\nMean in-plasma R²:", plasma_scores.mean().round(4))

Validation rows: 64523
In-plasma rows: 23401

In-plasma R² per beam property:


E_MeV      -3.5197
q_pC        0.9214
dE_pct      0.1726
emit_um    -0.5103
div_mrad   -0.0379
sigz_um     0.5022
dtype: float64


Mean in-plasma R²: -0.412


## 6. Checking the Longitudinal Grid

In [19]:
# Examine longitudinal positions for three simulations
configs = T_clean["config"].unique()[:3]

for config in configs:
    data = T_clean[T_clean["config"] == config]

    print(f"\nConfiguration {config}")
    print("Number of positions:", len(data))
    print("First five z positions:")
    print(data["z_mm"].sort_values().head().to_numpy())


Configuration 1000
Number of positions: 113
First five z positions:
[2.015055 2.060053 2.105055 2.150053 2.195155]

Configuration 1005
Number of positions: 113
First five z positions:
[2.015055 2.060053 2.105055 2.150053 2.195155]

Configuration 1008
Number of positions: 113
First five z positions:
[2.015055 2.060053 2.105055 2.150053 2.195155]


## 7. Position-Dependent R² Evaluation

In [20]:
# Check the longitudinal grid across configurations
grid_check = T_clean.groupby("config")["z_mm"].apply(
    lambda z: tuple(np.round(np.sort(z.to_numpy()), 5))
)

print("Number of distinct longitudinal grids:", grid_check.nunique())

# Build validation table
eval_df = X_val.copy()

for target in TARGETS:
    eval_df[target] = y_val[target]
    eval_df[f"pred_{target}"] = y_pred[:, TARGETS.index(target)]

# Identify the in-plasma region
eval_df["in_plasma"] = (
    eval_df["z_mm"] <= eval_df["L_inj"] + 3.2
)

plasma_df = eval_df[eval_df["in_plasma"]].copy()

# Group by actual longitudinal position
plasma_df["z_position"] = plasma_df["z_mm"].round(5)

position_scores = []

for z, data in plasma_df.groupby("z_position"):

    if len(data) < 2:
        continue

    scores = {"z_mm": z}

    for target in TARGETS:
        scores[target] = r2_score(
            data[target],
            data[f"pred_{target}"]
        )

    position_scores.append(scores)

position_df = pd.DataFrame(position_scores)

# Calculate mean R² across positions
plasma_scores = position_df[TARGETS].mean()

display(
    pd.DataFrame({
        "In-plasma R²": plasma_scores
    }).round(4)
)

print("Mean in-plasma R²:", plasma_scores.mean().round(4))

Number of distinct longitudinal grids: 1


,In-plasma R²
E_MeV,-3.5197
q_pC,0.9214
dE_pct,0.1726
emit_um,-0.5103
div_mrad,-0.0379
sigz_um,0.5022


Mean in-plasma R²: -0.412


## 8. Neural Network — Trajectory Prediction

We train a Multi-Layer Perceptron to predict six beam
properties along the accelerator.

The model has three hidden layers containing 256, 128,
and 64 neurons.

Inputs and outputs are standardized.

We compare its in-plasma performance against the
Linear Regression baseline.

In [21]:
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor

# Define MLP
mlp_model = TransformedTargetRegressor(
    regressor=make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(256, 128, 64),
            activation="relu",
            solver="adam",
            learning_rate_init=0.0005,
            alpha=0.001,
            max_iter=300,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=30,
            random_state=42
        )
    ),
    transformer=StandardScaler()
)

# Train
mlp_model.fit(X_train, y_train)

# Predict
mlp_pred = mlp_model.predict(X_val)

print("MLP training completed.")

MLP training completed.


## Compare in-plasma R²

In [22]:
# Copy validation inputs
mlp_eval = X_val.copy()

# Add true and predicted beam properties
for i, target in enumerate(TARGETS):
    mlp_eval[target] = y_val[target]
    mlp_eval[f"pred_{target}"] = mlp_pred[:, i]

# Keep only in-plasma measurements
mlp_eval = mlp_eval[
    mlp_eval["z_mm"] <= mlp_eval["L_inj"] + 3.2
].copy()

# Group by longitudinal position
mlp_eval["z_position"] = mlp_eval["z_mm"].round(5)

# Calculate R² at each position
mlp_position_scores = []

for z, data in mlp_eval.groupby("z_position"):

    if len(data) < 2:
        continue

    scores = {"z_mm": z}

    for target in TARGETS:
        scores[target] = r2_score(
            data[target],
            data[f"pred_{target}"]
        )

    mlp_position_scores.append(scores)

mlp_position_df = pd.DataFrame(mlp_position_scores)

# Mean in-plasma R²
mlp_plasma_scores = mlp_position_df[TARGETS].mean()

# Compare models
comparison = pd.DataFrame({
    "Linear Regression": plasma_scores,
    "MLP": mlp_plasma_scores
})

comparison.loc["mean"] = comparison.mean()

display(comparison.round(4))

,Linear Regression,MLP
E_MeV,-3.5197,0.9947
q_pC,0.9214,0.9979
dE_pct,0.1726,0.9764
emit_um,-0.5103,0.9865
div_mrad,-0.0379,0.9861
sigz_um,0.5022,0.9782
mean,-0.4120,0.9866
